# <font color='blue'>Data Science Academy</font>
# <font color='blue'>Deep Learning Para Aplicações de IA com PyTorch e Lightning</font>

## <font color='blue'>Estudo de Caso 4</font>
## <font color='blue'>Geração de Faces Humanas Realísticas com Redes Generativas</font>

![DSA](images/EC4.png)

## Instalando e Carregando os Pacotes

In [ ]:
# Versão da Linguagem Python
from platform import python_version
print('Versão da Linguagem Python Usada Neste Jupyter Notebook:', python_version())

In [ ]:
# Para atualizar um pacote, execute o comando abaixo no terminal ou prompt de comando:
# pip install -U nome_pacote

# Para instalar a versão exata de um pacote, execute o comando abaixo no terminal ou prompt de comando:
# !pip install nome_pacote==versão_desejada

# Depois de instalar ou atualizar o pacote, reinicie o jupyter notebook.

# Instala o pacote watermark. 
# Esse pacote é usado para gravar as versões de outros pacotes usados neste jupyter notebook.
#!pip install -q -U watermark

In [ ]:
#!pip install -q torch==2.0.0
!pip install -q torch

In [ ]:
#!pip install -q torchvision==0.15.1
!pip install -q torchvision

In [ ]:
# Imports
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.utils.data
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

In [ ]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "Data Science Academy" --iversions

## Definindo Parâmetros Globais

In [ ]:
# Seed para reproduzir o mesmo resultado
manualSeed = 999
random.seed(manualSeed)
torch.manual_seed(manualSeed)

Baixe os dados em: http://mmlab.ie.cuhk.edu.hk/projects/CelebA.html

In [ ]:
# Diretório raiz dos dados
dataroot = "data/celeba"

In [ ]:
# Número de workers para o dataloader
workers = 2

In [ ]:
# Número de GPUs disponíveis. Use 0 para o modo de CPU.
#ngpu = 1
ngpu = 0

## Preparando a Pasta de Imagens e o Dataloader

In [ ]:
# Todas as imagens serão redimensionadas para este tamanho usando um transformador
image_size = 64

In [ ]:
# Cria o dataset com a pasta de imagens
dataset = dset.ImageFolder(root = dataroot,
                           transform = transforms.Compose([
                               transforms.Resize(image_size),
                               transforms.CenterCrop(image_size),
                               transforms.ToTensor(),
                               transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
                           ]))

In [ ]:
# Batch size para o treinamento
batch_size = 128

In [ ]:
# Cria o dataloader
dataloader = torch.utils.data.DataLoader(dataset, 
                                         batch_size = batch_size,
                                         shuffle = True, 
                                         num_workers = workers)

In [ ]:
# Device usado no treinamento
device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

In [ ]:
print(device)

## Visualizando Imagens de Treino

In [ ]:
# Plot de algumas imagens de treino
real_batch = next(iter(dataloader))
plt.figure(figsize = (8,8))
plt.axis("off")
plt.title("Imagens de Treino")
plt.imshow(np.transpose(vutils.make_grid(real_batch[0].to(device)[:64], 
                                         padding = 2, 
                                         normalize = True).cpu(),(1,2,0)))

## Modelagem com Generative Adversarial Networks (GANs)

Generative Adversarial Networks (GANs) são uma classe de algoritmos de aprendizado de máquina inventados por Ian Goodfellow e seus colegas em 2014. As GANs são essencialmente dois modelos de rede neural que são treinados simultaneamente e competem um com o outro em um jogo de adversários.

O primeiro modelo é o Gerador (G), que tenta criar dados que parecem ter vindo do conjunto de dados original. O segundo modelo é o Discriminador (D), que tenta distinguir entre exemplos reais do conjunto de dados e exemplos criados pelo Gerador. 

![DSA](images/GAN.png)

O Gerador é treinado para maximizar a probabilidade de o Discriminador cometer um erro, enquanto o Discriminador é treinado para melhorar a sua capacidade de distinguir entre dados reais e falsos. Portanto, os dois modelos estão em competição um com o outro, daí o nome "adversário".

Esse método foi uma inovação significativa no campo do aprendizado de máquina e desde então tem sido usado para uma ampla gama de aplicações, incluindo a geração de imagens realistas, aprimoramento de resolução de imagem, entre outras.

Vale ressaltar que as GANs, como muitos modelos de aprendizado de máquina, têm limitações. Por exemplo, o treinamento de GANs pode ser instável e às vezes pode resultar em geração de dados de baixa qualidade. Além disso, a qualidade dos dados gerados pode ser difícil de avaliar objetivamente.

Os modelos GAN são considerados como sendo de aprendizado não-supervisionado.

Paper das GANs: https://arxiv.org/abs/1406.2661

## Construindo Arquitetura Deep Convolutional Generative Adversarial Networks (DCGANs) 

![DSA](images/DCGAN.png)

Deep Convolutional Generative Adversarial Networks (DCGANs) são uma extensão das Generative Adversarial Networks (GANs) que usam camadas convolucionais em vez de apenas camadas completamente conectadas. Elas foram propostas por Radford et al. em 2015, na tentativa de melhorar a qualidade e a estabilidade do treinamento das GANs.

Uma DCGAN consiste em duas partes principais: o Gerador (G) e o Discriminador (D).

O Gerador (G) é uma rede que recebe um vetor de ruído aleatório como entrada e o transforma em uma imagem via deconvolução (também conhecida como convolução transposta). O objetivo do Gerador é enganar o Discriminador produzindo imagens que pareçam vir do conjunto de dados original.

O Discriminador (D) é uma rede convolucional que recebe uma imagem como entrada (pode ser uma imagem real do conjunto de dados ou uma imagem gerada pelo Gerador) e produz uma única saída, representando a probabilidade de a imagem de entrada ser real (em vez de falsa). O objetivo do Discriminador é distinguir corretamente entre imagens reais e falsas.

Na arquitetura DCGAN, existem várias diretrizes de design que foram propostas para ajudar a estabilizar o treinamento e a produzir imagens de maior qualidade:

1- Substituir qualquer camada de pooling por strided convolutions (discriminador) e convoluções fracionadas (gerador): Isso permite que a rede aprenda a fazer sua própria downsampling/upsampling.

2- Usar batch normalization em ambas as redes: Isso ajuda a estabilizar o aprendizado removendo a mudança de covariância interna.

3- Remover as camadas totalmente conectadas para profundidade maior que 4: Isso permite que a rede seja mais profundamente composta de camadas convolucionais.

4- Usar ReLU na camada de saída do gerador: Isso ajuda a adicionar não-linearidade à rede.

5- Usar LeakyReLU no discriminador: Isso ajuda a evitar o problema de gradientes que desaparecem.

As DCGANs têm sido usadas em várias aplicações interessantes como geração de imagens, transferência de estilo, aprendizado de representações não supervisionadas, entre outras.

Paper original da arquitetura: https://arxiv.org/abs/1511.06434

In [ ]:
# Inicialização customizada dos pesos nas redes dsanetG e dsanetD
def init_pesos(m):
    
    classname = m.__class__.__name__
    
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

In [ ]:
# Número de canais de cores nas imagens de treinamento
nc = 3

In [ ]:
# Tamanho do vetor latente z (ou seja, tamanho da entrada do gerador)
nz = 100

In [ ]:
# Tamanho dos mapas de recursos no gerador
ngf = 64

In [ ]:
# Tamanho dos mapas de recursos no discriminador
ndf = 64

## Construindo o Generator

In [ ]:
# Classe do Generator
class Generator(nn.Module):
    
    # Método Construtor
    def __init__(self, ngpu):
        
        super(Generator, self).__init__()
        
        self.ngpu = ngpu
        
        self.main = nn.Sequential(
            
            # Input 
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias = False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            
            # Size: (ngf*8) x 4 x 4
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias = False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            
            # Size: (ngf*4) x 8 x 8
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias = False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            
            # Size: (ngf*2) x 16 x 16
            nn.ConvTranspose2d( ngf * 2, ngf, 4, 2, 1, bias = False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            
            # Size: (ngf) x 32 x 32
            nn.ConvTranspose2d( ngf, nc, 4, 2, 1, bias = False),
            nn.Tanh()
        )

    # Método Forward
    def forward(self, input):
        return self.main(input)

In [ ]:
# Cria o gerador
dsanetG = Generator(ngpu).to(device)

In [ ]:
# Inicializa os pesos
dsanetG.apply(init_pesos)

In [ ]:
# Print do modelo
print(dsanetG)

## Construindo o Discriminator

In [ ]:
# Classe do Discriminator
class Discriminator(nn.Module):
    
    # Método Construtor
    def __init__(self, ngpu):
        
        super(Discriminator, self).__init__()
        
        self.ngpu = ngpu
        
        self.main = nn.Sequential(
            
            # Input é (nc) x 64 x 64
            nn.Conv2d(nc, ndf, 4, 2, 1, bias = False),
            nn.LeakyReLU(0.2, inplace = True),
            
            # Size: (ndf) x 32 x 32
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias = False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace = True),
            
            # Size: (ndf*2) x 16 x 16
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias = False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace = True),
            
            # Size: (ndf*4) x 8 x 8
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias = False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace = True),
            
            # Size: (ndf*8) x 4 x 4
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias = False),
            nn.Sigmoid()
        )

    # Método Forward
    def forward(self, input):
        return self.main(input)

In [ ]:
# Cria o discriminador
dsanetD = Discriminator(ngpu).to(device)

In [ ]:
# Inicializa os pesos
dsanetD.apply(init_pesos)

In [ ]:
# Print do modelo
print(dsanetD)

## Função de Perda e Otimizador

In [ ]:
# Inicializa a função de erro BCELoss
criterion = nn.BCELoss()

A função nn.BCELoss() no PyTorch é a função de Perda de Entropia Cruzada Binária (Binary Cross Entropy Loss). É comumente usada em problemas de classificação binária, ou seja, quando a tarefa é prever uma das duas classes possíveis (0 ou 1).

A entropia cruzada binária é definida da seguinte maneira: 

BCELoss = -[y * log(p) + (1 - y) * log(1 - p)]

Onde:

- y é a classe verdadeira (0 ou 1).
- p é a probabilidade prevista pela rede neural para a classe 1.

A rede neural aprende ajustando seus parâmetros para minimizar a perda, neste caso a BCELoss. Portanto, a rede está tentando ajustar suas previsões p para ficarem mais próximas das classes verdadeiras y.

In [ ]:
# Taxa de aprendizado
lr = 0.0002

lr é a taxa de aprendizado ou learning rate. É um dos hiperparâmetros mais importantes em qualquer algoritmo de otimização de aprendizado de máquina. A taxa de aprendizado controla o quão rápido o modelo é atualizado em resposta ao gradiente da função de perda. Uma taxa de aprendizado muito alta pode fazer o modelo convergir muito rapidamente, mas pode pular o mínimo global e resultar em uma solução subótima. Por outro lado, uma taxa de aprendizado muito baixa pode fazer com que o modelo demore muito para convergir, ou pode ficar preso em um mínimo local.

In [ ]:
# Hiperparâmetros do otimizador Adam 
beta1 = 0.5
beta2 = 0.999

betas é um par de hiperparâmetros que são usados para calcular as estimativas de primeiro e segundo momentos do gradiente (semelhantes à média móvel e variância móvel, respectivamente). Os momentos são usados para atualizar os parâmetros do modelo. Por padrão, betas é definido como (0.9, 0.999). O primeiro valor em betas é o decaimento exponencial da taxa para a estimativa do primeiro momento (média móvel) e o segundo valor é o decaimento exponencial da taxa para a estimativa do segundo momento (variância móvel). Ambos os valores devem estar no intervalo [0, 1).

In [ ]:
# Otimizador Adam para ambas as redes G e D
optimizerD = optim.Adam(dsanetD.parameters(), lr = lr, betas = (beta1, beta2))
optimizerG = optim.Adam(dsanetG.parameters(), lr = lr, betas = (beta1, beta2))

## Loop de Treinamento e Avaliação

In [ ]:
# Número de passadas de treino
num_epochs = 5

In [ ]:
# Estabelece convenções para rótulos reais e falsos durante o treinamento
real_label = 1.
fake_label = 0.

In [ ]:
# Cria um lote de vetores latentes que usaremos para visualizar a progressão do gerador (para avaliação do modelo)
fixed_noise = torch.randn(64, nz, 1, 1, device = device)

A função torch.randn() é usada para gerar uma matriz com números aleatórios normalmente distribuídos (com média 0 e variância 1). Os argumentos para essa função definem as dimensões dessa matriz.

No caso de torch.randn(64, nz, 1, 1, device = device), isso cria uma matriz 4D de tamanho (64, nz, 1, 1). Vamos quebrar esses valores:

64: Este é o tamanho do batch. Ou seja, serão gerados 64 "exemplos" diferentes.

nz: Este é o tamanho da dimensão de características latentes (ou seja, o "tamanho" de cada exemplo). Nos GANs, o vetor latente é um vetor de números aleatórios que é fornecido como entrada para o gerador. A dimensão desse vetor é um hiperparâmetro que você pode definir.

1, 1: Estas são as dimensões espaciais da matriz. No caso de imagens, as duas primeiras dimensões normalmente se referem à altura e largura da imagem. Nesse caso, como estamos apenas criando um vetor de entrada para o gerador, essas dimensões são 1.

device=device: Especifica o dispositivo onde a matriz será armazenada, que pode ser a CPU ou a GPU. 

In [ ]:
type(fixed_noise)

In [ ]:
fixed_noise.size()

In [ ]:
fixed_noise

Em modelos de Deep Convolutional Generative Adversarial Networks (DCGANs), e outras variações de Generative Adversarial Networks (GANs), vetores latentes são vetores de números aleatórios que são a entrada para o gerador.

O Gerador em uma DCGAN é uma rede neural que tem como objetivo criar novas imagens que se pareçam com o conjunto de dados de treinamento. O gerador recebe como entrada um vetor latente - um vetor de números aleatórios - e o transforma, através de uma série de camadas convolucionais transpostas (ou deconvoluções), em uma imagem.

Vetores latentes: O espaço latente é essencialmente o espaço dos vetores de entrada para o gerador. Cada ponto neste espaço corresponde a uma possível imagem de saída. Vetores diferentes no espaço latente darão origem a imagens diferentes quando passarem pelo gerador.

Esses vetores latentes são fundamentais para o funcionamento das DCGANs e outras GANs. Eles fornecem o elemento de aleatoriedade que permite que o gerador crie uma variedade de imagens diferentes. Além disso, porque o gerador aprende a mapear o espaço latente para o espaço de imagens durante o treinamento, pequenas alterações nos vetores latentes podem levar a pequenas alterações nas imagens de saída. Isso é útil para tarefas como a interpolação de imagens, onde você quer gerar imagens que mudam suavemente de uma imagem para outra.

In [ ]:
# Listas para acompanhar o progresso
img_list = []
G_losses = []
D_losses = []
iters = 0

In [ ]:
%%time
print("Iniciando o Treinamento...")

# Loop por cada época
for epoch in range(num_epochs):
    
    # Loop por cada batch do dataloader
    for i, data in enumerate(dataloader, 0):

        ############################
        # (1) Atualiza a rede D maximizando: log(D(x)) + log(1 - D(G(z)))
        ###########################
        
        # Zera os gradientes
        dsanetD.zero_grad()
        
        # Prepara o batch de dados
        real_cpu = data[0].to(device)
        b_size = real_cpu.size(0)
        label = torch.full((b_size,), real_label, dtype = torch.float, device = device)
        
        # Forward pass de dados reais pela rede D
        output = dsanetD(real_cpu).view(-1)
        
        # Calcula o erro
        errD_real = criterion(output, label)
        
        # Calcula os gradientes da rede D no backward pass para os dados reais
        errD_real.backward()
        D_x = output.mean().item()

        # Gera o vetor latente
        noise = torch.randn(b_size, nz, 1, 1, device = device)
        
        # Gera imagens fake com a rede G
        fake = dsanetG(noise)
        label.fill_(fake_label)
        
        # Classifica as imagens fake com a rede D
        output = dsanetD(fake.detach()).view(-1)
        
        # Calcula o erro de D com as imagens fake
        errD_fake = criterion(output, label)
        
        # Calcula os gradientes para cada batch nos dados fake
        errD_fake.backward()
        D_G_z1 = output.mean().item()
        
        # O erro da rede D é a soma dos erros com dados reais e com dados fake
        errD = errD_real + errD_fake
        
        # Atualiza os pesos da rede D
        optimizerD.step()

        ############################
        # (2) Atualiza a rede G maximizando: log(D(G(z)))
        ###########################
        
        # Zera os gradientes
        dsanetG.zero_grad()
        
        # Labels fake são preenchidos com labels reais
        label.fill_(real_label)  
        
        # Como acabamos de atualizar D, executamos outra passagem de lote de imagens fake por D
        output = dsanetD(fake).view(-1)
        
        # Calculamos a perda de G com base nesta saída anterior
        errG = criterion(output, label)
        
        # Calculamos os gradientes de G
        errG.backward()
        D_G_z2 = output.mean().item()
        
        # Atualiza os pesos em G
        optimizerG.step()

        # Estatística de treino
        if i % 50 == 0:
            print('[%d/%d][%d/%d] Loss_D: %.4f Loss_G: %.4f'
                  % (epoch, num_epochs, i, len(dataloader),
                     errD.item(), errG.item()))

        # Salva os erros para criar um plot mais tarde
        G_losses.append(errG.item())
        D_losses.append(errD.item())

        # Verificamos como o gerador está salvando a saída de G em fixed_noise
        if (iters % 500 == 0) or ((epoch == num_epochs-1) and (i == len(dataloader)-1)):
            with torch.no_grad():
                fake = dsanetG(fixed_noise).detach().cpu()
                
            img_list.append(vutils.make_grid(fake, padding = 2, normalize = True))

        iters += 1
        
print("Treinamento Concluído!")

In [ ]:
# Plot
plt.figure(figsize = (10,5))
plt.title("Erro do Generator e Discriminator Durante o Treinamento")
plt.plot(G_losses,label = "G")
plt.plot(D_losses,label = "D")
plt.xlabel("Iterações")
plt.ylabel("Erro")
plt.legend()
plt.show()

In [ ]:
# Animação
fig = plt.figure(figsize=(8,8))
plt.axis("off")
ims = [[plt.imshow(np.transpose(i,(1,2,0)), animated = True)] for i in img_list]
ani = animation.ArtistAnimation(fig, ims, interval = 1000, repeat_delay = 1000, blit = True)
HTML(ani.to_jshtml())

In [ ]:
# Obtém um lote de imagens reais do dataloader
real_batch = next(iter(dataloader))

In [ ]:
# Plot das imagens reais
plt.figure(figsize = (15,15))
plt.subplot(1,2,1)
plt.axis("off")
plt.title("Imagens Reais")
plt.imshow(np.transpose(vutils.make_grid(real_batch[0].to(device)[:64], 
                                         padding = 5, 
                                         normalize = True).cpu(),(1,2,0)))

# Plot das imagens fake na última época de treino
plt.subplot(1,2,2)
plt.axis("off")
plt.title("Imagens Fake")
plt.imshow(np.transpose(img_list[-1],(1,2,0)))
plt.show()

## Fim